In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")[:10]
OPENAI_API_KEY

'sk-proj-zd'

In [3]:
# 필수 import v1.0
from langchain_openai.chat_models.base import ChatOpenAI
from langchain_openai.llms.base import OpenAI
from langchain_core.output_parsers.base import BaseOutputParser
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate

In [4]:
chat = ChatOpenAI()

In [8]:
result = chat.invoke("독도는 어느나라 땅이야?")

In [9]:
result.content

"독도는 한국의 땅이다. 해당 지역은 일본에서 '竹島(たけしま)'이라고 불리고 있지만, 한국에서는 '독도(독도)'로 불린다. 독도는 한국의 영토적 주장과 일본의 영토적 주장 사이에 논란이 있는 지역으로, 현재 한국 정부가 관리하고 있는 곳이다."

In [ ]:
# 같은 프롬프트 라도 결과를 다르게 할 수 있을까?
# llm 의 창의력 때문에 동일한 프롬프트 라도 다르게 나올 수 있다 (템퍼레이쳐)

## 창의력
- 0.0 ~ 0.3 결정론, 요약, 추출, 일관성, 정확성
- 0.5 ~ 0.7 : 적당히 자연스럽고 유연함
- 0.8 ~ 1이상: 무작위 : 창의적이 됨 예측 불가능

In [76]:
chat = ChatOpenAI(temperature=0)

In [10]:
chat = ChatOpenAI(temperature=0)

result = chat.invoke("하늘은 무슨 색이니?")

result.content

'하늘은 파란색이에요.'

In [11]:
chat = ChatOpenAI(temperature = 0.5)

result2 = chat.invoke("하늘은 무슨 색이니?")

result2.content

'하늘은 파란색이에요.'

In [17]:
chat = ChatOpenAI(temperature = 1.0)

result3 = chat.invoke("하늘은 무슨 색이니?")

result3.content

'하늘의 색은 보통 푸르게 보입니다. 하지만 일출이나 일몰 시간에는 붉은 빛이 감도는 경우도 있습니다. 때에 따라 하늘의 색은 다양하게 변할 수 있습니다.'

In [24]:
class NewLineOutputParser(BaseOutputParser):
    def parse(self, text):
        lines = text.split("\n")
        return [line.lstrip(" -123456789. ") for line in lines]

In [25]:
newline_parser = NewLineOutputParser()

In [26]:
newline_parser.parse("""
    - 1. 햄버거
    - 2. 피자
""")

['', '햄버거', '피자', '']

In [27]:
templet = ChatPromptTemplate.from_messages([
    ("system", """리스트를 생성하는 기계입니다.
        요청한 모든 리스트에 개수는 최대 {max_length}개 까지만 목록으로 표시하세요.
        그 이상 초과되는 리스트는 답변하지 마세요."""),
    ("human", "{question}")
])

In [28]:
prompt = templet.format_messages(
    max_length = 5,
    question = "AI를 잘하려면 어떤 것부터 공부해야 되?"
)

# 체인 생성
- "|" : 파이프 연산자로 체인을 만든다

In [31]:
# list
first_chain = templet | chat | NewLineOutputParser()

In [32]:
chain_result = first_chain.invoke({
    "max_length": 5,
    "question": "AI를 잘하려면 어떤 것부터 공부해야 되?"
})

In [33]:
chain_result

['머신러닝', '딥러닝', '자연어 처리', '컴퓨터 비전', '강화학습']

In [34]:
# 이전 단계의 출력이 다음 단계 입력으로 자동 전달되는 파이프라인 객체
print(type(first_chain))

<class 'langchain_core.runnables.base.RunnableSequence'>


## 1. 템플릿 생성

In [37]:
templet = ChatPromptTemplate.from_messages([
    ("system", """
        당신은 세계적인 수준의 여행 가이드 입니다.
        사람들이 좋아하는 여행 명소를 많이 알고 있습니다.
        설명없이 지역 이름만 목록으로 {max_length}개 까지 답변하시오
        목록의 갯수가 초과하는것은 답변하지 마세요
    """),
    ("human", """
        {place}여행 명소 추천해줘
    """)
])

In [38]:
second_chain = templet | chat

In [39]:
trip_result = second_chain.invoke({
    "max_length": 5,
    "place": "대전"
})

In [41]:
trip_result.content

'1. 유성온천\n2. 대청호\n3. 대전 엑스포 과학공원\n4. 한밭수목원\n5. 대전 문화예술단지'

# 실습

In [42]:
# 저녁 메뉴 추천(양식, 중식, 한식 선택할 수 있도록)
# 체인을 생성 후 결과를 출력

In [50]:
templet = ChatPromptTemplate.from_messages([
    ("system", """
        당신은 서울시 강남구 혹은 서울시 강서구에 있는 다양한 식당을 알고 있습니다.
        사용자는 1인 식사 (혼밥) 을 하는 상황이고 예산은 {price}원 입니다.
        지역, 이름, 간단한 설명 형태의 목록으로 {max_length}개 까지 답변하세요.
        목록의 갯수를 초과하는것은 답변하지 마세요
    """),
    ("human", "오늘 {type} 타입의 저녁 메뉴 추천해줘.")
])

In [44]:
templet

ChatPromptTemplate(input_variables=['max_length', 'price', 'type'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['max_length', 'price'], input_types={}, partial_variables={}, template='\n        당신은 서울시 강남구 혹은 강서구에 있는 다양한 식당을 알고 있습니다.\n        사용자는 1인 식사 (혼밥) 을 하는 상황이고 예산은 {price}원 입니다.\n        지역, 이름, 간단한 설명 형태의 목록으로 {max_length}개 까지 답변하세요.\n        목록의 갯수를 초과하는것은 답변하지 마세요\n    '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['type'], input_types={}, partial_variables={}, template='오늘 {type} 타입의 저녁 메뉴 추천해줘.'), additional_kwargs={})])

In [51]:
menu_chain = templet | chat

In [54]:
menu_result = menu_chain.stream({
    "price": 10000,
    "max_length": 5,
    "type": "한식"
})

In [53]:
menu_result.content

'다음은 서울시 강남구와 강서구에서 1인 식사로 즐길 수 있는 한식 식당 목록입니다:\n\n1. 강남구 - "남도한식당"\n   - 남도 지방의 전통 음식을 맛볼 수 있는 식당으로, 해장하기 좋은 음식들이 준비되어 있습니다.\n\n2. 강서구 - "한우마을"\n   - 한우로 유명한 식당으로, 신선한 한우고기와 한정식을 1만원 이내에 즐길 수 있습니다.\n\n이 외에도 다른 지역이나 더 많은 식당 정보를 원하시면 말씀해주세요!'

# stream
- 결과를 실시간 chunk 단위로 쪼개서 응답한다

In [55]:
for chunk in menu_result:
    print(chunk.content, end = "", flush=True)

서울시 강남구와 강서구에 있는 한식 식당 추천드릴게요:

1. 강남구 - "부자집"
   - 한정식 메뉴가 다양하며, 직접 만든 장을 사용해 고급스러운 한식을 즐길 수 있어요.

2. 강서구 - "별채한식당"
   - 한정식이 아닌 가격이 저렴한 한식당으로, 깔끔한 한식을 맛볼 수 있어요.

이 두 곳 중에서 선택하시면 10000원 이하의 예산으로도 맛있는 한식을 즐길 수 있을 거예요.

In [ ]:
# 최근 폭락 한 한국 주식 리스트 5개
# 딕셔너리

In [66]:
templete = ChatPromptTemplate.from_messages([
    ("system", """
        넌 주식의 전문 트레이더야
        사용자가 주식과 관련된 질문을 하면 {max_length}개의 주식을 추천해줘

        반드시 아래의 JSON 형식으로 답변해줘.
        출력 형식:
        {{
            "one": "삼천당제약",
            "two": "한미반도체",
            ...
            "five": "신라젠"
        }}
        
    """),
    ("human", "{question}")
])

In [67]:
templete

ChatPromptTemplate(input_variables=['max_length', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['max_length'], input_types={}, partial_variables={}, template='\n        넌 주식의 전문 트레이더야\n        사용자가 주식과 관련된 질문을 하면 {max_length}개의 주식을 추천해줘\n\n        반드시 아래의 JSON 형식으로 답변해줘.\n        출력 형식:\n        {{\n            "one": "삼천당제약",\n            "two": "한미반도체",\n            ...\n            "five": "신라젠"\n        }}\n\n    '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])

In [77]:
stock_chain = templete | chat

In [78]:
result = stock_chain.invoke({
    "max_length": 5,
    "question": "최근 대한민국 주식 중에서 급락한 주식 종목 뭔지 알려줘"
})

In [79]:
result.content

'{\n    "one": "한국전력",\n    "two": "대한항공",\n    "three": "현대차",\n    "four": "셀트리온",\n    "five": "SK하이닉스"\n}'

In [80]:
import json

In [81]:
json_result = json.loads(result.content)

json_result

{'one': '한국전력',
 'two': '대한항공',
 'three': '현대차',
 'four': '셀트리온',
 'five': 'SK하이닉스'}